# Feature Engineering
Building model-ready features for Attrition Prediction from the new real dataset.

In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/processed/employee_attrition_processed.csv')
print(f'Loaded: {df.shape}')
print(f'Attrition distribution: {df["Attrition_encoded"].value_counts().to_dict()}')

Loaded: (500, 29)
Attrition distribution: {0: 445, 1: 55}


In [2]:
# --- Engineered Features ---

# Feature 1: Income per year of tenure (identifies underpaid veterans)
if 'MonthlySalary' in df.columns and 'YearsAtCompany' in df.columns:
    df['IncomePerYear'] = df['MonthlySalary'] / (df['YearsAtCompany'] + 1)

# Feature 2: Overall satisfaction composite
if 'WorkLifeBalance' in df.columns and 'EngagementScore' in df.columns:
    df['OverallSatisfaction'] = (df['WorkLifeBalance'] + df['EngagementScore']) / 2.0
elif 'WorkLifeBalance' in df.columns:
    df['OverallSatisfaction'] = df['WorkLifeBalance']

# Feature 3: Experience ratio (tenure vs max possible career length)
if 'YearsAtCompany' in df.columns and 'Age' in df.columns:
    df['ExperienceRatio'] = df['YearsAtCompany'] / (df['Age'] - 18 + 1)

# Feature 4: Years since last promotion (career stagnation)
if 'LastPromotionYear' in df.columns:
    current_year = 2026
    df['YearsSincePromotion'] = current_year - df['LastPromotionYear'].fillna(current_year)

# Feature 5: Training investment index
if 'TrainingHours' in df.columns:
    df['TrainingIndex'] = df['TrainingHours'] / (df['YearsAtCompany'] + 1)

# Feature 6: KPI performance gap (if available)
if 'KPI_Score' in df.columns and 'PerformanceRating' in df.columns:
    df['PerformanceGap'] = df['KPI_Score'] - (df['PerformanceRating'] * 20)  # normalize rating to 0-100

print('Engineered features added.')
print([c for c in df.columns if c not in pd.read_csv('../data/processed/employee_attrition_processed.csv').columns])

Engineered features added.
['IncomePerYear', 'OverallSatisfaction', 'ExperienceRatio', 'YearsSincePromotion', 'TrainingIndex']


In [3]:
# --- Encode Categoricals ---
# Drop non-feature identifier and string columns
id_cols = ['EmployeeID', 'Name', 'AttritionRisk', 'JoiningDate', 'Country', 'Gender']
df_model = df.drop(columns=[c for c in id_cols if c in df.columns], errors='ignore')

# Encode string columns via one-hot
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'One-hot encoding: {cat_cols}')
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

print(f'Feature matrix shape: {df_encoded.shape}')
print(f'Target: Attrition_encoded -> {df_encoded["Attrition_encoded"].value_counts().to_dict()}')

os.makedirs('../data/processed/features', exist_ok=True)
df_encoded.to_csv('../data/processed/features/attrition_features.csv', index=False)
print('Features engineered and saved to data/processed/features/attrition_features.csv')

One-hot encoding: ['Department', 'JobRole']
Feature matrix shape: (500, 43)
Target: Attrition_encoded -> {0: 445, 1: 55}
Features engineered and saved to data/processed/features/attrition_features.csv
